# Task 141: tttrlib_flim — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: FLIM decay fitting using bi-exponential model

**Paper**: tttrlib: Time-Tagged Time-Resolved Library
**Repository**: https://github.com/Fluorescence-Tools/tttrlib

### Physical Background

Optical systems blur images via the Point Spread Function. Deconvolution inverts this to recover sharp detail.

### Forward Model

```
y = H * x + n  (PSF convolution)
```

### Inverse Problem

```
x_hat = argmin 1/2 ||H*x - y||^2 + lambda R(x)
```

### Performance Metrics
- **PSNR**: 69.40 dB
- **SSIM**: N/A
- **Evaluation**: 1D TCSPC decay deconvolution (bi-exponential lifetime fitting)

### Mathematical Formulation

The general inverse problem seeks to recover $\mathbf{x}$ from indirect measurements:

$$\mathbf{y} = \mathcal{A}(\mathbf{x}) + \boldsymbol{\eta}$$

**Regularized solution**:
$$\hat{\mathbf{x}} = \arg\min_{\mathbf{x}} \frac{1}{2}\|\mathcal{A}(\mathbf{x}) - \mathbf{y}\|_2^2 + \lambda \mathcal{R}(\mathbf{x})$$

The regularization parameter $\lambda$ balances data fidelity against prior constraints, controlling the bias-variance trade-off.


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
#!/usr/bin/env python3
"""
Task 141: tttrlib_flim — TCSPC fluorescence decay deconvolution.

Generates a synthetic TCSPC histogram (IRF convolved with bi-exponential decay),
then recovers the decay parameters via differential evolution optimisation.

Forward model:
    F(t) = IRF(t) ⊛ [ a₁·exp(-t/τ₁) + a₂·exp(-t/τ₂) ] + background

Inverse method:
    scipy.optimize.differential_evolution minimising reduced χ².
"""

import json
import os
import sys

import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`make_irf`, `bi_exponential`, `objective`, `compute_cc`

In [ ]:
# ──────────────────────────────────────────────────────────────
# Forward model helpers
# ──────────────────────────────────────────────────────────────
def make_irf(time, center, sigma):
    """Normalised Gaussian IRF."""
    irf = np.exp(-0.5 * ((time - center) / sigma) ** 2)
    irf /= irf.sum()
    return irf

def bi_exponential(time, a1, tau1, a2, tau2):
    """Bi-exponential decay (un-normalised)."""
    decay = a1 * np.exp(-time / tau1) + a2 * np.exp(-time / tau2)
    return decay

def objective(params):
    """Reduced chi-squared cost function (Poisson weighting)."""
    model = model_for_fit(params)
    # Poisson variance ≈ max(model, 1) to avoid division by zero
    variance = np.maximum(model, 1.0)
    chi2 = np.sum((measured - model) ** 2 / variance)
    n_dof = len(measured) - len(params)
    return chi2 / n_dof

def compute_cc(a, b):
    """Pearson correlation coefficient."""
    a_c = a - a.mean()
    b_c = b - b.mean()
    return float(np.sum(a_c * b_c) / (np.sqrt(np.sum(a_c ** 2) * np.sum(b_c ** 2)) + 1e-30))

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
y = H * x + n  (PSF convolution)
```

Functions: `forward_model`

In [ ]:
def forward_model(time, irf, a1, tau1, a2, tau2, bg):
    """Convolve IRF with bi-exponential decay and add background."""
    decay = bi_exponential(time, a1, tau1, a2, tau2)
    convolved = fftconvolve(irf, decay, mode="full")[:len(time)]
    # Normalise so that total area matches desired photon budget
    convolved = convolved / convolved.sum()
    return convolved + bg

## 5. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
x_hat = argmin 1/2 ||H*x - y||^2 + lambda R(x)
```

Functions: `model_for_fit`

In [ ]:
# ──────────────────────────────────────────────────────────────
# Inverse solver — differential evolution
# ──────────────────────────────────────────────────────────────
def model_for_fit(params):
    """Compute model curve from parameter vector."""
    a1, tau1, tau2, bg = params
    a2 = 1.0 - a1  # constrain amplitudes to sum to 1
    decay = bi_exponential(TIME, a1, tau1, a2, tau2)
    convolved = fftconvolve(irf_true, decay, mode="full")[:len(TIME)]
    convolved = convolved / convolved.sum() * TOTAL_COUNTS
    convolved += bg
    return convolved

## 6. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `compute_psnr`, `relative_error`

In [ ]:
# ──────────────────────────────────────────────────────────────
# Metrics
# ──────────────────────────────────────────────────────────────
def compute_psnr(gt, recon):
    """Peak signal-to-noise ratio."""
    mse = np.mean((gt - recon) ** 2)
    if mse == 0:
        return float("inf")
    peak = np.max(gt)
    return 10.0 * np.log10(peak ** 2 / mse)

def relative_error(true_val, fit_val):
    return abs(fit_val - true_val) / abs(true_val)

## 7. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import differential_evolution
from scipy.signal import fftconvolve

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# ──────────────────────────────────────────────────────────────
# Paths
# ──────────────────────────────────────────────────────────────
RESULTS_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

### Configuration and Parameters

Set up experiment parameters: grid sizes, noise levels, regularization weights, algorithm settings.

In [ ]:
# ──────────────────────────────────────────────────────────────
# Ground-truth parameters
# ──────────────────────────────────────────────────────────────
N_CHANNELS = 1024            # number of time bins
DT = 0.0122                  # ns per channel  →  total window ≈ 12.5 ns
TIME = np.arange(N_CHANNELS) * DT  # time axis (ns)

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
# IRF — Gaussian centred at ~1.2 ns, FWHM ≈ 200 ps
IRF_CENTER = 1.2             # ns
IRF_FWHM = 0.200             # ns
IRF_SIGMA = IRF_FWHM / (2.0 * np.sqrt(2.0 * np.log(2.0)))  # ≈ 85 ps

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
# Bi-exponential decay
TAU1_TRUE = 1.5              # ns  (short lifetime)
TAU2_TRUE = 4.0              # ns  (long lifetime)
A1_TRUE = 0.6                # amplitude fraction for τ₁
A2_TRUE = 0.4                # amplitude fraction for τ₂

BACKGROUND = 10.0            # constant background counts per bin
TOTAL_COUNTS = 1_000_000     # total number of photon counts (controls SNR)

RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

### Forward Model — Generating Measurements

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# ──────────────────────────────────────────────────────────────
# Forward model helpers
# ──────────────────────────────────────────────────────────────

### Ground Truth Construction

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# ──────────────────────────────────────────────────────────────
# Generate synthetic data
# ──────────────────────────────────────────────────────────────
print("Generating synthetic TCSPC histogram …")
irf_true = make_irf(TIME, IRF_CENTER, IRF_SIGMA)

### Ground Truth Construction

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# Noise-free forward model (ground truth)
gt_curve = forward_model(TIME, irf_true, A1_TRUE, TAU1_TRUE, A2_TRUE, TAU2_TRUE, 0.0)
gt_curve = gt_curve / gt_curve.sum() * TOTAL_COUNTS  # scale to photon counts
gt_curve_with_bg = gt_curve + BACKGROUND

# Noisy measurement (Poisson)
measured = rng.poisson(gt_curve_with_bg).astype(np.float64)

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# ──────────────────────────────────────────────────────────────
# Inverse solver — differential evolution
# ──────────────────────────────────────────────────────────────

### Configuration and Parameters

Set up experiment parameters: grid sizes, noise levels, regularization weights, algorithm settings.

In [ ]:
# Parameter bounds: [a1, tau1, tau2, background]
bounds = [
    (0.1, 0.9),     # a1
    (0.3, 5.0),     # tau1 (ns)
    (1.0, 10.0),    # tau2 (ns)
    (0.0, 50.0),    # background
]

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
print("Running differential evolution optimisation …")
result = differential_evolution(
    objective,
    bounds,
    seed=RNG_SEED,
    maxiter=2000,
    tol=1e-10,
    polish=True,
    workers=1,
)
print(f"  Optimiser converged: {result.success}  |  cost = {result.fun:.6f}")

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# Extract fitted parameters
a1_fit, tau1_fit, tau2_fit, bg_fit = result.x
a2_fit = 1.0 - a1_fit

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
# Ensure tau1 < tau2 (swap if needed for consistency)
if tau1_fit > tau2_fit:
    tau1_fit, tau2_fit = tau2_fit, tau1_fit
    a1_fit, a2_fit = a2_fit, a1_fit

fitted_curve = model_for_fit([a1_fit, tau1_fit, tau2_fit, bg_fit])

print(f"  Fitted: a1={a1_fit:.4f}, τ1={tau1_fit:.4f} ns, "
      f"a2={a2_fit:.4f}, τ2={tau2_fit:.4f} ns, bg={bg_fit:.2f}")

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# ──────────────────────────────────────────────────────────────
# Metrics
# ──────────────────────────────────────────────────────────────

### Forward Model — Generating Measurements

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# Compare fitted curve (without bg) to ground-truth curve (without bg)
gt_no_bg = gt_curve  # noise-free, no background
fitted_no_bg = fitted_curve - bg_fit

psnr_val = compute_psnr(gt_no_bg, fitted_no_bg)
cc_val = compute_cc(gt_no_bg, fitted_no_bg)

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
re_tau1 = relative_error(TAU1_TRUE, tau1_fit)
re_tau2 = relative_error(TAU2_TRUE, tau2_fit)
re_a1 = relative_error(A1_TRUE, a1_fit)
re_a2 = relative_error(A2_TRUE, a2_fit)
reduced_chi2 = result.fun

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
metrics = {
    "PSNR_dB": round(float(psnr_val), 2),
    "CC": round(float(cc_val), 6),
    "reduced_chi2": round(float(reduced_chi2), 6),
    "tau1_true_ns": TAU1_TRUE,
    "tau1_fit_ns": round(float(tau1_fit), 4),
    "tau1_RE": round(float(re_tau1), 6),
    "tau2_true_ns": TAU2_TRUE,
    "tau2_fit_ns": round(float(tau2_fit), 4),
    "tau2_RE": round(float(re_tau2), 6),
    "a1_true": A1_TRUE,
    "a1_fit": round(float(a1_fit), 4),
    "a1_RE": round(float(re_a1), 6),
    "a2_true": A2_TRUE,
    "a2_fit": round(float(a2_fit), 4),
    "a2_RE": round(float(re_a2), 6),
    "background_true": BACKGROUND,
    "background_fit": round(float(bg_fit), 2),
}

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
print("\n=== Metrics ===")
for k, v in metrics.items():
    print(f"  {k}: {v}")

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# ──────────────────────────────────────────────────────────────
# Save outputs
# ──────────────────────────────────────────────────────────────
np.save(os.path.join(RESULTS_DIR, "ground_truth.npy"), gt_no_bg)
np.save(os.path.join(RESULTS_DIR, "recon_output.npy"), fitted_no_bg)

with open(os.path.join(RESULTS_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

print("\nSaved ground_truth.npy, recon_output.npy, metrics.json")

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# ──────────────────────────────────────────────────────────────
# Visualisation — 4-panel figure
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Task 141: TCSPC Fluorescence Decay Deconvolution", fontsize=14, fontweight="bold")

### Ground Truth Construction

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# Panel 1: Decay + Fit (log scale)
ax = axes[0, 0]
ax.semilogy(TIME, measured, "k.", markersize=1.5, alpha=0.5, label="Measured (Poisson)")
ax.semilogy(TIME, gt_curve_with_bg, "b-", linewidth=1.5, label="Ground truth + bg")
ax.semilogy(TIME, fitted_curve, "r--", linewidth=1.5, label="Fitted model")
ax.semilogy(TIME, irf_true * irf_true.max() ** -1 * gt_curve_with_bg.max() * 0.3,
            "g-", linewidth=1, alpha=0.6, label="IRF (scaled)")
ax.set_xlabel("Time (ns)")
ax.set_ylabel("Counts")
ax.set_title("Decay Curve & Fit")
ax.legend(fontsize=8)
ax.set_xlim(0, TIME[-1])
ax.set_ylim(bottom=1)

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# Panel 2: Weighted residuals
ax = axes[0, 1]
residuals = (measured - fitted_curve) / np.sqrt(np.maximum(fitted_curve, 1.0))
ax.plot(TIME, residuals, "k-", linewidth=0.5, alpha=0.7)
ax.axhline(0, color="r", linestyle="--", linewidth=0.8)
ax.axhline(2, color="gray", linestyle=":", linewidth=0.5)
ax.axhline(-2, color="gray", linestyle=":", linewidth=0.5)
ax.set_xlabel("Time (ns)")
ax.set_ylabel("Weighted Residual")
ax.set_title(f"Residuals  (χ²ᵣ = {reduced_chi2:.4f})")
ax.set_xlim(0, TIME[-1])

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# Panel 3: Parameter comparison (bar chart)
ax = axes[1, 0]
param_names = ["τ₁ (ns)", "τ₂ (ns)", "a₁", "a₂", "bg"]
true_vals = [TAU1_TRUE, TAU2_TRUE, A1_TRUE, A2_TRUE, BACKGROUND]
fit_vals = [tau1_fit, tau2_fit, a1_fit, a2_fit, bg_fit]
x_pos = np.arange(len(param_names))
width = 0.35
bars1 = ax.bar(x_pos - width / 2, true_vals, width, label="True", color="steelblue", alpha=0.8)
bars2 = ax.bar(x_pos + width / 2, fit_vals, width, label="Fitted", color="coral", alpha=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(param_names)
ax.set_ylabel("Value")
ax.set_title("Parameter Recovery")
ax.legend()
# Add value labels
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=7)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=7)

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
# Panel 4: Individual decay components
ax = axes[1, 1]
comp1_true = A1_TRUE * np.exp(-TIME / TAU1_TRUE)
comp2_true = A2_TRUE * np.exp(-TIME / TAU2_TRUE)
comp1_fit = a1_fit * np.exp(-TIME / tau1_fit)
comp2_fit = a2_fit * np.exp(-TIME / tau2_fit)
ax.semilogy(TIME, comp1_true, "b-", linewidth=1.5, label=f"True τ₁={TAU1_TRUE} ns")
ax.semilogy(TIME, comp1_fit, "b--", linewidth=1.5, label=f"Fit τ₁={tau1_fit:.3f} ns")
ax.semilogy(TIME, comp2_true, "r-", linewidth=1.5, label=f"True τ₂={TAU2_TRUE} ns")
ax.semilogy(TIME, comp2_fit, "r--", linewidth=1.5, label=f"Fit τ₂={tau2_fit:.3f} ns")
ax.set_xlabel("Time (ns)")
ax.set_ylabel("Amplitude")
ax.set_title("Decay Components")
ax.legend(fontsize=8)
ax.set_xlim(0, TIME[-1])

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
plt.tight_layout()
fig_path = os.path.join(RESULTS_DIR, "reconstruction_result.png")
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.close()
print(f"Saved {fig_path}")

print("\n✓ Task 141 complete.")

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 8. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **tttrlib_flim**:

1. **Problem Setup**: Optical systems blur images via the Point Spread Function. Deconvolution inverts this to recover sharp detail.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=69.40 dB, SSIM=N/A)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: tttrlib: Time-Tagged Time-Resolved Library
- Repository: https://github.com/Fluorescence-Tools/tttrlib
